In [1]:
from sklearn.linear_model import LassoCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
live_avg = pd.read_csv("../data/samples/mega_processed/live_avg.csv")
live_avg_norm = pd.read_csv("../data/samples/mega_processed/live_avg_norm.csv")
live_avg_scaled = pd.read_csv("../data/samples/mega_processed/live_avg_scaled.csv")

In [3]:
to_keep = ['GPU Utilization (%)',
 'Power Draw (Watts)',
 'GPU Current Clock (MHz)',
 'Memory Allocation Used (MB)',
 'Current Time']
to_drop = ['Memory Utilization (%)', 'Time Delta', 'Iteration', 'GPU Clock Utilization', 'GPU Temp (°C)',]

In [4]:
y = live_avg.iloc[:, -2:-1]
y = y['Resonse Time'].tolist()
live_avg = live_avg.iloc[:, :-1]

In [5]:
live_avg_norm = live_avg_norm.drop(columns=['Unnamed: 0'])
live_avg = live_avg.drop(columns=['Unnamed: 0'])
live_avg_scaled = live_avg_scaled.drop(columns=['Unnamed: 0'])

In [6]:
live_avg_norm.columns = live_avg.columns[:-1]
live_avg_norm_aug = live_avg_norm.drop(columns=to_drop)
live_avg_norm = live_avg_norm.drop(columns=['Iteration'])

In [13]:
X_train, X_test, y_train, y_test = train_test_split(live_avg_norm, y, test_size=0.2, shuffle=True, random_state=20)

In [14]:
lcv = LassoCV(cv=3)
lcv.fit(X_train, y_train)
score = lcv.score(X_test, y_test)
print(f"Accuracy: {score}")
print(f"Weights: {lcv.coef_} & Alpha: {lcv.alpha_}")

Accuracy: 0.0936727978366122
Weights: [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 -6.89424143e+10
  1.78714262e+09  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -0.00000000e+00  0.00000000e+00] & Alpha: 2.267552220972199e-12


In [9]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import Lasso, Ridge

rfe = RFE(estimator=Lasso(), n_features_to_select=2, verbose=True)

In [19]:
rfe.fit(live_avg_norm.iloc[:325, :], y_train)
rfe.score(live_avg_norm.iloc[325:, :], y_test)

Fitting estimator with 10 features.
Fitting estimator with 9 features.
Fitting estimator with 8 features.
Fitting estimator with 7 features.
Fitting estimator with 6 features.
Fitting estimator with 5 features.
Fitting estimator with 4 features.
Fitting estimator with 3 features.


-0.001921412018106139

In [20]:
dict(zip(live_avg_norm.columns.tolist(), rfe.support_))

{'GPU Utilization (%)': False,
 'Power Draw (Watts)': False,
 'GPU Temp (°C)': False,
 'GPU Current Clock (MHz)': False,
 'Memory Allocation Used (MB)': False,
 'Memory Utilization (%)': False,
 'Current Time': False,
 'Time Delta': False,
 'GPU Clock Utilization': True,
 'Memory Clock Utilization': True}

In [21]:
# live_avg_norm_aug = live_avg_norm_aug.drop(columns=['Current Time'])
live_avg_norm_aug['Response Time'] = y
live_avg_norm_aug.to_csv('../data/samples/live_avg_norm_aug.csv')